# STEP 1 — Environment Setup & Dataset Preparation

**What this step does:**
1. Installs all required packages
2. Downloads and inspects all three datasets (GSM8K, BBH, TruthfulQA)
3. Generates paraphrases for Axis 3 (consistency) using a lightweight local model — saved to disk once, reused forever

**Outputs produced:**
- `data/datasets.json` — all questions with ground-truth answers
- `data/paraphrases.json` — 5 paraphrases per question (for consistency experiments)

**Run time:** ~10–15 min (paraphrase generation is the bottleneck)

**Once this step succeeds, you do NOT need to re-run it.**

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers==4.44.0 accelerate bitsandbytes datasets evaluate
!pip install -q pandas numpy matplotlib seaborn scipy scikit-learn
print('✓ Packages installed')

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/light_paper')  # adjust if running locally

import json
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR   = Path('/kaggle/working/light_paper')
DATA_DIR   = BASE_DIR / 'data'
RESULTS_DIR= BASE_DIR / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS_PATH    = DATA_DIR / 'datasets.json'
PARAPHRASES_PATH = DATA_DIR / 'paraphrases.json'

print(f'BASE_DIR: {BASE_DIR}')
print(f'DATA_DIR: {DATA_DIR}')

## 1a. Load Datasets

In [ ]:
from src.datasets import load_all_datasets

# Sizes: 200 GSM8K + ~180 BBH (3 subtasks × 60) + 200 TruthfulQA = ~580 questions
all_data = load_all_datasets(
    n_gsm8k=200,
    n_bbh_per_subtask=60,
    n_truthfulqa=200,
)

for ds_name, samples in all_data.items():
    print(f'  {ds_name}: {len(samples)} samples')

# Save to disk
with open(DATASETS_PATH, 'w') as f:
    json.dump(all_data, f, indent=2)
print(f'\n✓ Saved to {DATASETS_PATH}')

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────
print('GSM8K sample:')
s = all_data['gsm8k'][0]
print(f"  Q: {s['question'][:120]}...")
print(f"  A: {s['answer']}")

print('\nBBH sample:')
s = all_data['bbh'][0]
print(f"  Q: {s['question'][:120]}...")
print(f"  A: {s['answer']}")

print('\nTruthfulQA sample:')
s = all_data['truthfulqa'][0]
print(f"  Q: {s['question'][:120]}...")
print(f"  A: {s['answer']}")

## 1b. Generate Paraphrases

We use **Qwen2.5-3B-Instruct** locally to generate 5 paraphrases per question.
This runs once. Results are saved to `data/paraphrases.json`.

We only need paraphrases for GSM8K + TruthfulQA (400 questions total × 5 = 2000 paraphrases).

In [ ]:
from src.models import load_model, get_model_by_name
from src.prompts import make_paraphrase_gen_prompt
from src.inference import generate_responses
import re

# Use Qwen2.5-3B for paraphrase generation — good quality, fits on 1x T4
para_config = get_model_by_name('Qwen2.5-3B')
para_model, para_tokenizer = load_model(para_config)
print('✓ Paraphrase model loaded')

In [ ]:
def parse_paraphrases(raw: str) -> list[str]:
    """Extract numbered paraphrases from model output."""
    lines = [l.strip() for l in raw.split('\n') if l.strip()]
    paras = []
    for line in lines:
        m = re.match(r'^\d+[.)\s]+(.+)', line)
        if m:
            paras.append(m.group(1).strip())
    return paras[:5]  # exactly 5


# Questions needing paraphrases: GSM8K + TruthfulQA (not BBH — it's structured MCQ)
para_questions = all_data['gsm8k'] + all_data['truthfulqa']
print(f'Total questions to paraphrase: {len(para_questions)}')

# Check if already generated
if PARAPHRASES_PATH.exists():
    print('⚡ Paraphrases already exist — skipping generation. Load from disk:')
    with open(PARAPHRASES_PATH) as f:
        paraphrases = json.load(f)
    print(f'  Loaded {len(paraphrases)} entries')
else:
    paraphrases = {}
    prompts = [make_paraphrase_gen_prompt(q['question']) for q in para_questions]

    # Generate in batches
    raw_outputs = generate_responses(
        para_model, para_tokenizer, prompts,
        max_new_tokens=512, batch_size=4
    )

    for q, raw in zip(para_questions, raw_outputs):
        parsed = parse_paraphrases(raw)
        paraphrases[q['id']] = parsed

    # Save
    from src.datasets import save_paraphrases
    save_paraphrases(paraphrases, PARAPHRASES_PATH)
    print(f'✓ Generated & saved {len(paraphrases)} paraphrase sets')

In [ ]:
# ── Paraphrase quality check ──────────────────────────────────────────────
sample_id = all_data['gsm8k'][0]['id']
print(f"Original: {all_data['gsm8k'][0]['question'][:100]}...")
print(f"\nParaphrases:")
for i, p in enumerate(paraphrases.get(sample_id, [])):
    print(f"  {i+1}. {p[:100]}")

# Count how many questions have exactly 5 paraphrases
good = sum(1 for v in paraphrases.values() if len(v) == 5)
print(f"\nQuestions with exactly 5 paraphrases: {good}/{len(paraphrases)}")

In [ ]:
# Free GPU memory before next step
import gc, torch
del para_model, para_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('✓ GPU memory freed')
print('\n✅ STEP 1 COMPLETE')
print('   Outputs: data/datasets.json, data/paraphrases.json')
print('   Next: Run step2_accuracy.ipynb')